![](../assets/placeholder.png)

## Giriş
Bu notebook, Langchain, ChromaDB ve OpenAI'nin GPT-4 LLM'i kullanarak basit bir retrieval augmented generation uygulaması oluşturma sürecinde sizi yönlendirmek için tasarlanmıştır.

### RAG: Retrieval Augmented Generation (Almadan Güçlendirilmiş Üretim)
Retrieval Augmented Generation (RAG), dil modellerinin gücünü belirli içeriğe dayalı sorulara yanıt vermek için harici içerik retrieval ile birleştirir. Bu yaklaşım, bir bilgi veritabanından yararlanarak daha doğru ve bağlamla ilgili cevaplar sağlamaya olanak tanır.

Bu işlem birkaç temel adım içerir:
- **İçerik Retrieval ve Depolama**: İlk olarak, içeriğimizi alır ve aranabilir bir biçimde depolarız. Bu, bir URL'den içerik alınmasını, bunu yönetilebilir parçalara bölünmesini, bu parçaların anlamsal arama için gömülmesini ve bunları bir Vektör DB'de depolanmasını içerir.
- **Soru Cevaplama**: Bir kullanıcı sorusu sorulduğunda, sistem Vektör DB'den ilgili içeriği alır ve bunu LLM ile bir yanıt oluşturmak için kullanır.

Bu notebook, ChromaDB kullanan bir RAG sistemi kurmasını ve soruları cevaplamak için OpenAI'nin GPT-4'ünü kullanılmasını gösterecektir.

In [ ]:
from dotenv import load_dotenv
import os

# .env dosyasından ortam değişkenlerini yükleyin
load_dotenv()

In [ ]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
import bs4

# Blog içeriğini yükleyin, ayırın ve indeksleyin.
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)
docs = loader.load()

# İçeriği daha iyi retrieval için yönetilebilir parçalara ayırın.
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)

# Parçaları gömün ve verimli retrieval için ChromaDB'de depolayın.
vectorstore = Chroma.from_documents(documents=splits, embedding=OpenAIEmbeddings())

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

# Yanıtları almak ve oluşturmak için RAG zincirini ayarlayın.
retriever = vectorstore.as_retriever()
system_prompt = ("""
Soru cevaplama görevleri için bir asistansınız. Soruyu cevaplamak için alınan bağlamın aşağıdaki parçasını kullanın. Cevabı bilmiyorsanız, sadece bilmediğinizi söyleyin. En yüksek üç cümleyi kullanın ve cevabı kısa tutun.
Bağlam: {context}
""")

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{question}"),
    ]
)
                 

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

model = ChatOpenAI(model="gpt-4o")

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

In [ ]:
question = "Araç kullanımı nedir?"
rag_chain.invoke(question)